# Fase 6b — Pemetaan Topik → Aspek secara DATA-DRIVEN

Metode (saran dosen): **Sentence-BERT / e5 + cosine similarity**. Mengganti tebakan
subjektif topik→aspek dengan ukuran kuantitatif + bukti.

- Taksonomi aspek **berasal dari studi terpublikasi** (Maharani dkk. 2023, ITB+BRI;
  Twitter Financial News) — lihat `reports/justifikasi_aspek.md`.
- Output: `data/processed/topic_to_aspect_suggested.csv` = usulan otomatis + 3 kandidat
  teratas + `margin` (keyakinan). Anotator tinggal me-review (terutama margin kecil).

**Prasyarat:** sudah menjalankan Fase 6 (ada `bertopic_model/`). Pakai embedder yang sama (e5).

In [ ]:
import os
from pathlib import Path
import pandas as pd

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
import sys; sys.path.insert(0, str(ROOT))
from src.aspect_taxonomy import map_topics_to_aspects, ASPECT_DEFINITIONS
from src.topics import TopicConfig, detect_device

PROCESSED = ROOT / 'data/processed'
print('aspek kandidat (dari studi acuan):')
for c, d in ASPECT_DEFINITIONS.items():
    print(f'  {c}: {d["name"]}  <- {d["maps_from"]}')

In [ ]:
# Muat model BERTopic hasil Fase 6 + embedder yang SAMA (e5-base)
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

device = detect_device()
tcfg = TopicConfig()
embedder = SentenceTransformer(tcfg.embedding_model, device=device)
topic_model = BERTopic.load(str(PROCESSED / 'bertopic_model'), embedding_model=embedder)
print('model & embedder dimuat. device:', device)

In [ ]:
# Pemetaan data-driven
result = map_topics_to_aspects(topic_model, embedder, e5_prefix=tcfg.e5_prefix, top_k=3)
tbl = result.table

# tambahkan kolom untuk anotator (diisi manual, me-review usulan)
tbl['aspect_annotator_1'] = ''
tbl['aspect_annotator_2'] = ''
tbl['aspect_final'] = ''

out = PROCESSED / 'topic_to_aspect_suggested.csv'
tbl.to_csv(out, index=False)
print('tersimpan:', out.resolve())
print('kolom:', list(tbl.columns))
tbl.head(30)

In [ ]:
# Ringkasan: distribusi usulan aspek + topik AMBIGU (margin kecil) yg perlu perhatian anotator
print('=== distribusi usulan aspek ===')
print(tbl['aspect_suggested'].value_counts().to_string())
print('\n=== topik AMBIGU (margin < 0.05) — perlu review manual lebih teliti ===')
amb = tbl[tbl['margin'] < 0.05].sort_values('margin')
print(f'{len(amb)} dari {len(tbl)} topik ambigu')
for _, r in amb.head(15).iterrows():
    print(f'  T{r.topic_id}: {r.aspect_top1}({r.sim_top1}) vs {r.aspect_top2}({r.sim_top2}) | {r.keywords[:50]}')

## Langkah anotator (kamu + partner)

Buka `data/processed/topic_to_aspect_suggested.csv`:
- Kolom `aspect_suggested` = usulan otomatis (data-driven). `aspect_top1..3` + `sim_*` = bukti.
- Fokuskan review pada topik dengan **`margin` kecil** (ambigu) — di situ keputusan manusia paling penting.
- Isi `aspect_annotator_1` & `aspect_annotator_2` independen → rekonsiliasi `aspect_final`.
- Hitung Cohen's κ antar-anotator; target ≥ 0.70.

Metode ini memberi **dasar objektif** (bukan asumsi) + validasi manusia. Lihat `reports/justifikasi_aspek.md`.